# [센서 실습 3] 기압 센서 – BMP280

기압 센서 **BMP280**으로 48시간 동안 대기압을 측정하고, **3시간 기압 변화량**으로 날씨 악화를 예보하며, 기압으로 고도를 계산해 봅니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
BMP280 안에는 아주 얇은 실리콘 막(다이어프램)이 있어서, 공기가 누르는 힘에 따라 막이 휘면 **저항값(피에조 저항)** 이 변합니다.
- 칩 안에서 온도 보정까지 마친 디지털 값을 **I2C**(또는 SPI)로 보냄
- 높이 올라갈수록 공기가 희박해져 기압이 낮아짐 → **약 8m 오를 때마다 1hPa 감소** → 고도계로도 사용
- 일기 예보: 기압이 3시간에 3hPa 넘게 떨어지면 저기압 접근 → 비·바람 가능성 증가

### 부품 사양
| 항목 | 내용 |
|---|---|
| 측정 범위 | 300 ~ 1100 hPa |
| 상대 정확도 | ±0.12 hPa (고도 약 ±1m) |
| 절대 정확도 | ±1 hPa |
| 온도 | -40 ~ 85°C (±1°C) |
| 전원 | 1.7 ~ 3.6V (모듈은 3.3V) |
| 통신 | I2C 주소 0x76 또는 0x77 |

## 2. 라즈베리파이 연결 방법
**준비물**: BMP280 모듈(GY-BMP280), 점퍼선 4개

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| BMP280 VCC | 3.3V (1번 핀) |  |
| BMP280 GND | GND (6번 핀) |  |
| BMP280 SCL | GPIO3 / SCL (5번 핀) | I2C 클록 |
| BMP280 SDA | GPIO2 / SDA (3번 핀) | I2C 데이터 (주소 0x76) |

SDO 핀을 GND에 연결하면 0x76, 3.3V에 연결하면 0x77 주소가 됩니다.

**인터페이스 활성화**
- **I2C**: `sudo raspi-config` → Interface Options → **I2C** → Enable, 연결 확인: `i2cdetect -y 1`

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install adafruit-circuitpython-bmp280
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
국제 표준 대기 공식으로 기압을 고도로 바꿉니다. 해면 기압(`sea_level_hpa`)은 그날 기상청 값을 넣어야 정확합니다.

In [ ]:
def pressure_to_altitude(p_hpa, sea_level_hpa=1013.25):
    """국제 표준 대기 공식: 기압(hPa) → 해발 고도(m)"""
    return 44330.0 * (1 - (p_hpa / sea_level_hpa) ** (1 / 5.255))

for p in (1013.25, 1000, 900, 800):
    print(f"{p:7.2f} hPa → 약 {pressure_to_altitude(p):6.0f} m")

### 4-2. 실제 센서 클래스
Adafruit 라이브러리가 레지스터 읽기와 온도 보정 계산을 대신해 줍니다.

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealBMP280:
    def __init__(self, address=0x76):
        import board, adafruit_bmp280
        self.dev = adafruit_bmp280.Adafruit_BMP280_I2C(board.I2C(), address=address)

    def read(self):
        return {"pressure_hpa": round(self.dev.pressure, 2), "temp_c": round(self.dev.temperature, 2)}

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 항목 | 가정한 값 | 근거 |
|---|---|---|
| 평상시 기압 | 1015 hPa | 맑은 날 해면 기압 |
| 저기압 통과 | 20~36시간 사이 최대 −17 hPa 하강 후 회복 | 비구름을 동반한 저기압이 지나가는 상황 |
| 기압 조석 | ±1 hPa, 12시간 주기 | 대기가 하루 두 번 오르내리는 실제 현상 |
| 잡음 | ±0.12 hPa | 데이터시트 상대 정확도 |

In [ ]:
class VirtualBMP280:
    def __init__(self, dt=1800, seed=3):
        self.dt, self.t = dt, 0.0
        self.rng = np.random.default_rng(seed)

    def read(self):
        h = self.t / 3600
        p = 1015 + 1.0 * np.sin(2 * np.pi * h / 12)              # 기압 조석
        p -= 17 * np.exp(-((h - 30) / 6) ** 2)                    # 저기압 통과 (30시간째 최저)
        temp = 22 + 3 * np.sin(np.pi * ((h % 24) - 9) / 12)
        self.t += self.dt
        return {"pressure_hpa": round(p + self.rng.normal(0, 0.12), 2),
                "temp_c": round(temp + self.rng.normal(0, 0.3), 2)}

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualBMP280(dt=1800)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
30분 간격으로 48시간(96회) 측정합니다.

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 96               # 측정 횟수
INTERVAL = 1800               # 측정 간격(초)

sensor = RealBMP280() if USE_REAL_SENSOR else VirtualBMP280(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
**3시간 기압 변화량(기압 경향)** 으로 날씨를 판단합니다. 30분 간격이므로 6개 전 값과 비교합니다.

| 3시간 변화량 | 판정 |
|---|---|
| −3 hPa 이하 | 날씨 악화 예보 (저기압 접근) |
| +3 hPa 이상 | 날씨 회복 |
| 그 외 | 안정 |

In [ ]:
df["hour"] = df["time_s"] / 3600
steps_3h = int(3 * 3600 / INTERVAL)
df["trend_3h"] = df["pressure_hpa"].diff(steps_3h)
df["forecast"] = np.where(df["trend_3h"] <= -3, "악화", np.where(df["trend_3h"] >= 3, "회복", "안정"))

bad = df[df["forecast"] == "악화"]
if len(bad):
    print(f"날씨 악화 예보 최초 발생: {bad['hour'].iloc[0]:.1f}시간째 (3시간 동안 {bad['trend_3h'].iloc[0]:.1f} hPa)")
print(f"최저 기압 {df['pressure_hpa'].min():.1f} hPa ({df.loc[df['pressure_hpa'].idxmin(), 'hour']:.1f}시간째)")
print(f"현재 기압으로 계산한 고도: {pressure_to_altitude(df['pressure_hpa'].iloc[0]):.0f} m "
      "(해면 기압을 1013.25로 가정 – 날씨에 따라 오차가 생김)")

## 8. 시각화

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(df["hour"], df["pressure_hpa"], "k-", lw=1.5)
ax[0].set_ylabel("기압 (hPa)"); ax[0].grid(alpha=0.3); ax[0].set_title("BMP280 기압 측정 (48시간)")
ax[1].plot(df["hour"], df["trend_3h"], "k-", lw=1.5, label="3시간 변화량")
ax[1].axhline(-3, color="gray", ls="--", label="악화 기준 −3 hPa")
ax[1].fill_between(df["hour"], -3, df["trend_3h"], where=df["trend_3h"] <= -3, color="lightgray")
ax[1].set_ylabel("hPa / 3h"); ax[1].set_xlabel("경과 시간 (시간)"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "압력센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "BMP280",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/pressure", payload)
# client.disconnect()

## 직접 해 보기
- 학교 건물 1층과 4층에서 측정했다고 가정하고(약 12m 차이), 기압 차이가 얼마나 날지 계산해 보세요.
- 저기압의 세기(−17)를 −30으로 바꿔 태풍 상황을 만들어 보세요.
- 오늘 기상청 해면 기압을 `sea_level_hpa`에 넣으면 고도 계산이 얼마나 정확해지나요?